# Pobranie funduszu płac klubów z fbref

Wejście finansowe dla `02_panel_cre_probit.ipynb`: roczny fundusz płac każdego klubu
czterech lig w sezonach 2013/14 – 2023/24. Wynik zapisuje się do
`data/raw/place_top4_ligi_2013_2023.csv`.

> **Ten notebook nie był uruchamiany przy porządkowaniu repozytorium.** Plik
> `place_top4_ligi_2013_2023.csv` w `data/raw/` pochodzi z pierwotnych przebiegów autora
> i to on jest wejściem do analizy. Zapisana wcześniej wersja scrapera obsługiwała
> **tylko Premier League** i zapisywała plik o innej nazwie oraz innym układzie kolumn,
> więc nie odtwarzała pliku, z którego korzysta projekt. Poniższa wersja jest uogólniona
> na cztery ligi i produkuje dokładnie ten układ kolumn, który czyta
> `src/data_load.load_wages`. Przed ponownym uruchomieniem warto sprawdzić, czy
> identyfikatory rozgrywek i wzorce adresów fbref nadal są aktualne — serwis zmienia
> układ stron.

**Dlaczego przeglądarka, a nie `requests`:** fbref stoi za Cloudflare, a tabele płacowe
są w kodzie strony ukryte w komentarzach HTML. Potrzebne jest więc środowisko
wykonujące skrypty strony oraz usunięcie znaczników komentarza przed parsowaniem.

**Obciążenie serwera:** między zapytaniami jest przerwa (`REQUEST_DELAY`). Przy czterech
ligach i jedenastu sezonach to 44 zapytania, czyli około dwunastu minut samego
oczekiwania. Nie skracaj tego bez potrzeby.

In [ ]:
# Zaleznosci tego notebooka nie sa w requirements.txt projektu: sluza wylacznie
# jednorazowemu pobraniu danych, a analiza dziala na zapisanym pliku CSV.
# Odkomentuj przy pierwszym uruchomieniu.
# %pip install seleniumbase lxml

In [ ]:
import sys
import time
from io import StringIO
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()
while not (_here / 'src' / 'data_load.py').is_file():
    if _here == _here.parent:
        raise RuntimeError(f'Nie znaleziono korzenia projektu startujac z {Path.cwd()}.')
    _here = _here.parent
sys.path.insert(0, str(_here))

from src.data_load import RAW, SEASON_MAX, SEASON_MIN, WAGE_FILE

print('PROJECT_ROOT:', _here)
print('Plik docelowy:', WAGE_FILE)

## Konfiguracja

Identyfikatory rozgrywek i fragmenty adresów fbref. **Do sprawdzenia przed ponownym
uruchomieniem** — fbref zmienia numerację i układ adresów.

In [ ]:
# liga -> (identyfikator rozgrywek fbref, fragment adresu strony plac)
FBREF_LEAGUES = {
    'Premier_League': (9, 'Premier-League-Wages'),
    'Bundesliga': (20, 'Bundesliga-Wages'),
    'La_Liga': (12, 'La-Liga-Wages'),
    'Serie_A': (11, 'Serie-A-Wages'),
}

# Przerwa miedzy zapytaniami - zabezpieczenie przed blokada po stronie serwisu.
REQUEST_DELAY = 15

# Czas na wykonanie skryptow strony po jej otwarciu.
PAGE_SETTLE = 6

# Liczba prob ponownego polaczenia przy blokadzie Cloudflare.
RECONNECT_TRIES = 5

# Okno widoczne (headless=False) dziala pewniej przy obejsciu Cloudflare.
HEADLESS = False

# Kolumny pliku wynikowego - czyta je src.data_load.load_wages, wiec ich nazwy
# i kolejnosc nie sa dowolne.
OUTPUT_COLUMNS = ['League', 'Season', 'Squad', 'Weekly Wages', 'Annual Wages']

seasons = [f'{y}-{y + 1}' for y in range(SEASON_MIN, SEASON_MAX + 1)]
print(f'Sezony ({len(seasons)}):', seasons)
print('Zapytan do wykonania:', len(seasons) * len(FBREF_LEAGUES))

## Funkcje pomocnicze

Trzy kroki rozdzielone, żeby dało się je sprawdzić osobno: zbudowanie adresu, wyłuskanie
tabeli płacowej ze strony i oczyszczenie kwot.

In [ ]:
def wage_url(comp_id: int, slug: str, season: str) -> str:
    """Adres strony plac danej ligi i sezonu."""
    return f'https://fbref.com/en/comps/{comp_id}/{season}/wages/{season}-{slug}'


def find_wage_table(html: str) -> pd.DataFrame | None:
    """Tabela plac ze strony fbref.

    Tabele sa na stronie ukryte w komentarzach HTML, wiec przed parsowaniem
    usuwamy znaczniki komentarza. Z wszystkich tabel bierzemy pierwsza, ktora
    ma kolumne z kwotami plac.
    """
    cleaned = html.replace('<!--', '').replace('-->', '')
    for table in pd.read_html(StringIO(cleaned)):
        names = ' '.join(str(c) for c in table.columns)
        if 'Annual Wages' in names or 'Weekly Wages' in names:
            return table
    return None


def to_amount(series: pd.Series) -> pd.Series:
    """Kwota z komorki fbref na liczbe.

    Komorka ma postac '£ 2,177,962 (€ 2,540,000)' - bierzemy czesc przed
    nawiasem, czyli kwote w walucie lokalnej ligi, i usuwamy symbole oraz
    separatory tysiecy. Waluta rozni sie miedzy ligami, ale w modelu wchodzi
    logarytm, a stala przelicznika jest pochlaniana przez efekt staly ligi.
    """
    return pd.to_numeric(
        series.astype(str)
        .str.split('(').str[0]
        .str.replace(r'[^0-9.]', '', regex=True)
        .str.strip(),
        errors='coerce',
    )


def clean_wage_table(table: pd.DataFrame, league: str, season: str) -> pd.DataFrame:
    """Tabela fbref sprowadzona do ukladu pliku wynikowego."""
    if isinstance(table.columns, pd.MultiIndex):
        table = table.copy()
        table.columns = table.columns.droplevel(0)

    df = table[table['Squad'].notna()].copy()
    # Wiersze zbiorcze ('Squad Total', 'Average') nie sa klubami.
    df = df[~df['Squad'].str.contains('Total|Average', case=False, na=False)]

    df['Weekly Wages'] = to_amount(df['Weekly Wages'])
    df['Annual Wages'] = to_amount(df['Annual Wages'])
    df['League'] = league
    df['Season'] = season
    return df[OUTPUT_COLUMNS]

## Pobranie

Pętla po ligach i sezonach. Błąd pojedynczej strony nie przerywa całości — brakujące
pary liga × sezon są wypisane na końcu, żeby dało się je dociągnąć osobno.

In [ ]:
from seleniumbase import Driver

frames, failures = [], []
driver = Driver(uc=True, headless=HEADLESS)
try:
    for league, (comp_id, slug) in FBREF_LEAGUES.items():
        for season in seasons:
            url = wage_url(comp_id, slug, season)
            print(f'{league} {season}: {url}')
            try:
                driver.uc_open_with_reconnect(url, RECONNECT_TRIES)
                time.sleep(PAGE_SETTLE)
                table = find_wage_table(driver.page_source)
                if table is None:
                    raise ValueError('nie znaleziono tabeli plac na stronie')
                df = clean_wage_table(table, league, season)
                frames.append(df)
                print(f'   pobrano {len(df)} klubow')
            except Exception as exc:
                failures.append((league, season, repr(exc)))
                print(f'   BLAD: {exc}')
            time.sleep(REQUEST_DELAY)
finally:
    driver.quit()

print(f'\nPobrane strony: {len(frames)}; nieudane: {len(failures)}')
for league, season, exc in failures:
    print(f'   {league} {season}: {exc}')

## Zapis

Plik zapisuje się dopiero, gdy wszystkie pary liga × sezon zostały pobrane. Zapis
częściowego zestawu nadpisałby kompletne dane niekompletnymi i nic by tego nie
zasygnalizowało.

In [ ]:
expected = len(seasons) * len(FBREF_LEAGUES)
if len(frames) < expected:
    raise RuntimeError(
        f'Pobrano {len(frames)} z {expected} stron. Plik NIE zostal zapisany - '
        'dociagnij brakujace pary liga x sezon i powtorz te komorke.')

panel = pd.concat(frames, ignore_index=True)
print('Obserwacji:', len(panel))
print('Braki w kwotach:')
print(panel[['Weekly Wages', 'Annual Wages']].isna().sum().to_string())
print()
print(panel.groupby('League')['Season'].nunique().to_string())
panel.head(10)

In [ ]:
RAW.mkdir(parents=True, exist_ok=True)
panel.to_csv(WAGE_FILE, index=False, encoding='utf-8')
print('Zapisano:', WAGE_FILE, f'({len(panel)} obserwacji)')